o explain the math behind the calibration code, we focus on the relationship between individual forward rate volatilities (the "building blocks") and the aggregate volatility of a swap rate (the "target").
1. The Volatility Identity

The core assumption in LFM is that each forward rate Li​(t) follows log-normal dynamics. The market prices of Caplets are used to determine the "marginal" volatility of each rate.

We define σi​ as the constant instantaneous volatility of the i-th forward rate. Under LFM, the variance of the i-th forward rate over its life Ti​ is given by:
Vcaplet,i​=∫0Ti​​σi2​dt=σi2​Ti​

This σi​ is what you obtained from your Bootstrap (the "Spot Volatility").
2. The Swap Rate Approximation

A swap rate Sm,n​(t) is a weighted average of forward Libor rates:
Sm,n​(t)=i=m∑n−1​wi​(t)Li​(t)

where wi​(t) are time-dependent weights. To calibrate to Swaptions, we use the Rebonato Approximation, which relates the volatility of the swap rate (σswap​) to the volatilities (σi​,σj​) and correlations (ρij​) of the forward rates:
σswap2​≈i=m∑n−1​j=m∑n−1​wi​wj​Sm,n​(0)2Li​(0)Lj​(0)​σi​σj​ρij​

In the Python code, we simplified the weights (wi​) to demonstrate the principle. The key takeaway is that Swap Volatility is the square root of the double-sum of the covariance matrix.
3. Parametric Correlation Model

To avoid having too many parameters (which leads to overfitting), we assume a functional form for ρij​. The most common is the Exponential Decay model:
ρij​=e−β∣Ti​−Tj​∣

    If β=0, then ρij​=1 (Perfect correlation). The swap volatility would simply be the weighted average of forward volatilities.

    If β→∞, then ρij​=0 (Zero correlation). The swap volatility would be much lower due to the diversification effect.

4. The Optimization Logic

The "calibration" is actually a search for the parameter β that minimizes the distance between the Market and the Model.
The Objective Function:
βmin​f(β)=​σMkt_Swaption​−i,j∑​wi​wj​σi​σj​e−β∣Ti​−Tj​∣​​2
The Steps in the Code:

    Initialize: Pick a starting β (e.g., 0.1).

    Construct Matrix: Create the ρ matrix using the current β.

    Compute Model Vol: Sum the weighted covariances.

    Compare: Check the squared difference against the market target.

    Iterate: The scipy.optimize solver (Nelder-Mead) adjusts β until the difference is effectively zero.

Slide Summary for Students

    Caplets define the "Diagonal" of the covariance matrix (individual variances).

    Swaptions define the "Off-Diagonal" elements (correlations).

    The Beta (β): Controls how quickly correlation drops as we move further apart on the yield curve. A β calibrated to the market ensures that the model reflects the curve's historical or implied "tendency to twist."

These are excellent technical refinements. Transitioning from a "toy" model to a "real-world" LFM requires addressing precisely these three points.
1. Functional Form for σi​(t)

In the simple bootstrap, you get a "piecewise constant" volatility. However, using a functional form (parametric) is preferred because it ensures the volatility of a specific forward rate evolves smoothly as it approaches maturity.

The industry standard is the Rebonato Form:
σi​(t)=[a(Ti​−t)+b]e−c(Ti​−t)+d

    Why use it? It captures the "humped" shape often seen in interest rate volatility (where volatility increases for mid-term tenors and then decays).

    Calibration: Instead of calibrating 20 different σi​ values, you calibrate just 4 parameters (a,b,c,d) to fit all Caplet prices. This prevents "overfitting" and ensures that your Monte Carlo simulation doesn't have "jumps" in volatility as time passes.

2. Setting wi​ in the Rebonato Approximation

In a real-world scenario, the weights wi​ represent the sensitivity of the Swap Rate to each underlying Forward Rate. They are derived from the definition of the Swap Rate:
Sm,n​(t)=∑k=mn−1​δk​P(t,Tk+1​)P(t,Tm​)−P(t,Tn​)​

The weights used in the approximation are:
wi​=∑k=mn−1​δk​P(0,Tk+1​)δi​P(0,Ti+1​)​

Wait—there is a detail: The "true" Rebonato weights actually include a small adjustment for the Libor rates themselves, but in practice, the discount-bond-weighted approach above is the standard for calibration because it is stable and easy to compute from the initial yield curve.
3. Two-Factor Correlation Model

The one-factor model (e−β∣Ti​−Tj​∣) is often "too stiff." It assumes that if the 1Y and 2Y rates move together, then the 1Y and 10Y rates must follow a very specific decay. In reality, the long end of the curve often moves independently of the short end.

We use a Two-Factor Model to allow for a "Long-Term Correlation Floor":
ρij​=L+(1−L)e−β∣Ti​−Tj​∣

    L (Long-term level): The minimum correlation between any two rates, even if they are very far apart (e.g., L=0.4).

    β (Decay): How fast we reach that floor.


In [ ]:
def objective_two_factor(params):
    beta, L = params
    if beta < 0 or L < 0 or L > 1: return 1e10
    
    # Construct Two-Factor Rho
    rho = np.full((n, n), L)
    for i in range(n):
        for j in range(n):
            if i == j:
                rho[i, j] = 1.0
            else:
                rho[i, j] = L + (1 - L) * np.exp(-beta * abs(tenors[i] - tenors[j]))
    
    # Rebonato Approx
    sum_val = 0
    for i in range(n):
        for j in range(n):
            sum_val += weights[i] * weights[j] * caplet_vols[i] * caplet_vols[j] * rho[i, j]
    
    model_vol = np.sqrt(sum_val)
    return (model_vol - target_swaption_vol)**2

# Calibration with two parameters
res = minimize(objective_two_factor, [0.1, 0.4], method='Nelder-Mead')
beta_calib, L_calib = res.x

New Slide: One-Factor vs. Two-Factor Correlation

    One-Factor (e−βΔT):

        Pros: Simple, one parameter.

        Cons: Forces correlation to 0 for tenors far apart. The yield curve "bends" too much.

    Two-Factor (L+(1−L)e−βΔT):

        Pros: Captures "Parallel Shifts" (via L) and "Twists" (via β).

        Cons: Requires more market data (at least two different swaptions) to calibrate both parameters uniquely.

In [ ]:
import numpy as np
from scipy.optimize import minimize

# 1. Inputs from Bootstrap (Example Data)
# Tenors: 0.5Y, 1.0Y, 1.5Y, 2.0Y
tenors = np.array([0.5, 1.0, 1.5, 2.0])
delta = 0.5
caplet_vols = np.array([0.22, 0.20, 0.19, 0.18])

# 2. Market Target for Correlation
# Let's say a 2-year Swaption (into a 2-year swap) has a market vol of 16%
target_swaption_vol = 0.165 
weights = np.array([0.25, 0.25, 0.25, 0.25]) # Simplified swap weights (w_i)

# 3. Correlation Function (Parametric)
def get_rho_matrix(beta, n):
    rho = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            rho[i, j] = np.exp(-beta * abs(tenors[i] - tenors[j]))
    return rho

# 4. Calibration Objective Function
def objective_function(beta):
    if beta < 0: return 1e10 # Constraint: beta must be positive
    
    n = len(caplet_vols)
    rho = get_rho_matrix(beta, n)
    
    # LFM Approximation for Swap Volatility:
    # Var(Swap) = Sum_i Sum_j (w_i * w_j * sigma_i * sigma_j * rho_ij)
    sum_val = 0
    for i in range(n):
        for j in range(n):
            sum_val += weights[i] * weights[j] * caplet_vols[i] * caplet_vols[j] * rho[i, j]
    
    model_swaption_vol = np.sqrt(sum_val)
    return (model_swaption_vol - target_swaption_vol)**2

# 5. Run Calibration
initial_beta = 0.1
result = minimize(objective_function, initial_beta, method='Nelder-Mead')
calibrated_beta = result.x[0]

# 6. Final Outputs
print(f"--- Calibration Results ---")
print(f"Calibrated Beta (Decay Factor): {calibrated_beta:.4f}")
print(f"Final Correlation (Short vs Long Tenor): {np.exp(-calibrated_beta * (tenors[-1]-tenors[0])):.4f}")

# Re-calculate to check match
final_rho = get_rho_matrix(calibrated_beta, len(caplet_vols))
final_vol = np.sqrt(np.sum(np.outer(weights * caplet_vols, weights * caplet_vols) * final_rho))
print(f"Target Swaption Vol: {target_swaption_vol:.4f}")
print(f"Model Swaption Vol:  {final_vol:.4f}")

In [ ]:
import numpy as np
from scipy.optimize import minimize

def calibrate_lfm(tenors, market_caplet_vols, target_swaption_vol, weights):
    """
    Jointly calibrates Rebonato Volatility Form and Two-Factor Correlation.
    
    tenors: array of maturities (e.g., 0.5, 1.0, ...)
    market_caplet_vols: bootstrapped spot vols for each tenor
    target_swaption_vol: market vol for the specific swaption
    weights: Rebonato weights (w_i) calculated from yield curve
    """
    
    # --- STEP 1: Calibrate Rebonato Volatility Form ---
    # sigma(tau) = [a*tau + b] * exp(-c*tau) + d
    def vol_objective(params):
        a, b, c, d = params
        # Avoid negative volatilities
        model_vols = (a * tenors + b) * np.exp(-c * tenors) + d
        return np.sum((model_vols - market_caplet_vols)**2)

    # Initial guess for [a, b, c, d]
    vol_init = [0.1, 0.1, 0.5, 0.1]
    vol_res = minimize(vol_objective, vol_init, method='Nelder-Mead')
    a_f, b_f, c_f, d_f = vol_res.x
    
    # Final fitted sigma_i values
    fitted_sigmas = (a_f * tenors + b_f) * np.exp(-c_f * tenors) + d_f

    # --- STEP 2: Calibrate Two-Factor Correlation ---
    # rho(i,j) = L + (1-L) * exp(-beta * |Ti - Tj|)
    def corr_objective(params):
        beta, L = params
        # Constraints: beta > 0, 0 <= L <= 1
        if beta < 0 or L < 0 or L > 1: return 1e10
        
        n = len(tenors)
        # Vectorized construction of rho
        T_i, T_j = np.meshgrid(tenors, tenors)
        rho = L + (1 - L) * np.exp(-beta * np.abs(T_i - T_j))
        np.fill_diagonal(rho, 1.0) # Ensure diagonal is exactly 1
        
        # Rebonato Approximation for Swap Vol
        # Using matrix multiplication for efficiency: w*sigma * rho * (w*sigma)^T
        sig_w = weights * fitted_sigmas
        model_swap_var = sig_w @ rho @ sig_w.T
        return (np.sqrt(model_swap_var) - target_swaption_vol)**2

    # Initial guess for [beta, L]
    corr_init = [0.1, 0.5]
    corr_res = minimize(corr_objective, corr_init, method='Nelder-Mead')
    beta_f, L_f = corr_res.x

    return {
        "vol_params": {"a": a_f, "b": b_f, "c": c_f, "d": d_f},
        "corr_params": {"beta": beta_f, "L": L_f},
        "fitted_sigmas": fitted_sigmas
    }

# --- Example Usage ---
tenors = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
mkt_vols = np.array([0.25, 0.24, 0.22, 0.21, 0.20, 0.19])
target_swap_vol = 0.17
# Example weights (sum to 1)
w = np.array([0.1, 0.15, 0.2, 0.2, 0.2, 0.15])

results = calibrate_lfm(tenors, mkt_vols, target_swap_vol, w)

print(f"Fitted Vol Params: {results['vol_params']}")
print(f"Fitted Corr Params: {results['corr_params']}")

In [23]:
import numpy as np
from scipy.stats import norm
from scipy.linalg import cholesky

# 1. Parameters
n_paths = 5000
n_steps = 10
T_expiry = 1.0
delta = 0.5
num_rates = 6
L0 = np.full(num_rates, 0.05)
sigmas = np.full(num_rates, 0.20)
K = 0.05
dt = T_expiry / n_steps

# 2. Define Correlation Matrix (Rho)
# Example: Constant correlation decaying with distance between tenors
rho = np.zeros((num_rates, num_rates))
beta = 0.1 # Decay factor
for i in range(num_rates):
    for j in range(num_rates):
        rho[i, j] = np.exp(-beta * abs(i - j))

# Cholesky Decomposition: R = A * A.T
A = cholesky(rho, lower=True)

# 3. Simulation Setup
L = np.tile(L0, (n_paths, 1)) # (n_paths, num_rates)

# 4. Monte Carlo Loop
for t in range(n_steps):
    # Generate Independent Normals
    Z = np.random.normal(0, 1, (n_paths, num_rates))
    
    # Correlate them: dW = Z * A.T (Linear transformation)
    # This results in dW having the correlation structure 'rho'
    dW = Z @ A.T * np.sqrt(dt)
    
    L_new = L.copy()
    for i in range(num_rates - 1):
        # Calculate Correlated Drift
        drift_sum = 0
        for j in range(i + 1, num_rates):
            term = (delta * L[:, j] * sigmas[i] * sigmas[j] * rho[i, j]) / (1 + delta * L[:, j])
            drift_sum -= term
        
        # Euler Step
        # dLi = drift*dt + sigma*Li*dWi
        L_new[:, i] += drift_sum * dt + sigmas[i] * L[:, i] * dW[:, i]

    last = num_rates - 1
    L_new[:, last] += 0 + sigmas[last] * L[:, last] * dW[:, last]
    
    L = L_new

# 5. Pricing & Control Variate (Same logic as before)
S_T = np.mean(L[:, :4], axis=1) # Swap rate estimate
payoff_lfm = np.maximum(S_T - K, 0)

# # Black-76 for the first rate as CV
# Use the LAST rate (index -1) because it has NO drift under terminal measure
last_idx = num_rates - 1

# 1. Analytical Black Price for the LAST rate
d1 = (np.log(L0[last_idx]/K) + 0.5 * sigmas[last_idx]**2 * T_expiry) / (sigmas[last_idx] * np.sqrt(T_expiry))
d2 = d1 - sigmas[last_idx] * np.sqrt(T_expiry)
black_price = (L0[last_idx] * norm.cdf(d1) - K * norm.cdf(d2))

# 2. Path payoff for that SAME last rate
payoff_black_path = np.maximum(L[:, last_idx] - K, 0)

correlation = np.corrcoef(payoff_lfm, payoff_black_path)[0, 1]
print(f"Correlation between LFM and Control: {correlation:.4f}")

# Calculate Beta Opt
beta_opt = np.cov(payoff_lfm, payoff_black_path)[0, 1] / np.var(payoff_black_path)

# Calculate Prices
price_raw = np.mean(payoff_lfm)
price_cv_opt = np.mean(payoff_lfm + beta_opt * (black_price - payoff_black_path))

print(f"Raw Price: {price_raw:.6f}")
print(f"Optimized CV Price: {price_cv_opt:.6f}")
print (np.var(payoff_lfm))
print (np.var(payoff_lfm + beta_opt * (black_price - payoff_black_path)))
print(f"New Variance Reduction: {np.var(payoff_lfm) / np.var(payoff_lfm + beta_opt * (black_price - payoff_black_path)):.2f}x")

Correlation between LFM and Control: 0.6543
Raw Price: 0.002457
Optimized CV Price: 0.002460
2.4925058822903162e-05
1.4253497556928183e-05
New Variance Reduction: 1.75x
